# QuantumTrain 工具書03：有梯度卻學不會

透過20筆過擬合測試與affine/Tanh對照，學習定位訓練問題。

## 0. 模型有更新，為什麼沒學會？

**本節目標**：區分更新正確與任務有效。

**為什麼需要這一步**：第一輪 QT 約10.2%，現在診斷而非再次搜尋 test 最佳設定。

**輸入／輸出與 shape**：(B,64) → 610 weights → hidden (B,8) → logits (B,10)。

**關鍵原理與程式定位**：本輪固定 depth3，研究 affine-only mapping 與 Tanh mapping。所有數據只來自 train/validation。

**操作步驟**：先執行 setup，保持 smoke；正式模式使用 full。

**預期觀察**：顯示 CPU、float64 與模式。

**結果判讀與常見錯誤**：這是第二轮探索；原測試集已被看過，不能當全新的盲測。

**理解問題**：loss.backward 成功是否等於學會？

<details><summary>提示與解答</summary>

不等於。還需要看訓練資料是否可擬合、logits 是否區分輸入及驗證表現。

</details>


In [ ]:
import os, sys, json, time
from pathlib import Path
import torch, pennylane as qml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from qt_eval.config import load_config, validate_config
from qt_eval.data import load_digits_splits
from qt_eval.models import make_model, ClassicalMLP, functional_mlp
from qt_eval.quantum import QuantumTrain, make_qnode, basis_codes, probability_features
from qt_eval.diagnostics import check_training_step
from qt_eval.experiment import run_experiment, evaluate
from qt_eval.reporting import summarize_runs, write_report
from qt_eval.artifacts import SOURCE_COMMIT

# 教學預設 smoke；只有準備好正式預算後才手動改成 full。
MODE = os.environ.get('QT_EVAL_MODE', 'smoke')
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '請由專案根目錄或 notebooks 目錄啟動'
OUTPUT = Path(os.environ.get('QT_EVAL_OUTPUT', str(ROOT/'results'/'notebooks')))
CONFIG = load_config(ROOT/'configs'/('smoke.json' if MODE=='smoke' else 'depth_study.json'))
assert CONFIG['mode'] == MODE
torch.set_num_threads(CONFIG['torch_threads'])
torch.manual_seed(CONFIG['seed'])
print('MODE:', MODE, '| device: cpu | dtype: float64')
if MODE == 'smoke':
    print('工程檢查：小資料、短訓練，不可用來判斷模型研究表現。')

## 1. 先看輸出與隱藏層

**本節目標**：檢查是否塌縮成同一類。

**為什麼需要這一步**：平均 loss 看不到預測集中在哪裡。

**輸入／輸出與 shape**：固定40筆 train probe → 10類計數、8個hidden活躍比例與權重/logits統計。

**關鍵原理與程式定位**：snapshot 不 backward、不消耗RNG，資料固定才能前後比較；ReLU負輸入變成0。

**操作步驟**：看初始化預測圖與統計，再把圖和後續訓練結果比較。

**預期觀察**：可能出現初始化偏向單一類；這本身不是bug證明。

**結果判讀與常見錯誤**：全部或大部分 ReLU=0 可能妨礙學習，但要配合梯度看，不能單憑一張圖定因。

**理解問題**：logits在不同圖片之間幾乎不變意味什麼？

<details><summary>提示與解答</summary>

目前模型沒有很好區分輸入；檢查權重與activation，不能只調電路深度。

</details>


In [ ]:
from qt_eval.training_diagnostics import snapshot,run_mapping_study
data=load_digits_splits(smoke=MODE=='smoke')
probe=data['train'].tensors[0][:40]
torch.manual_seed(11)
model=QuantumTrain(3)
initial=snapshot(model,probe)
display(initial)
plt.bar(range(10),initial['prediction_counts']); plt.xlabel('Predicted digit'); plt.ylabel('Count on fixed train probe'); plt.show()
print('圖說：固定訓練probe的預測類別數量，不是測試準確率。')

## 2. 只改 Mapping 的非線性

**本節目標**：建立受控的兩組模型。

**為什麼需要這一步**：多個Linear相乘仍是affine，生成610個權重的表達能力可能受限。

**輸入／輸出與 shape**：兩組皆 (610,11) → (610,1)，相同16,761個mapping參數。

**關鍵原理與程式定位**：Tanh加在前三個Linear之後，最後層不加。重設相同seed，使量子與各Linear初始權重逐一相同。

**操作步驟**：列印兩組架構並逐一assert初始參數一致。

**預期觀察**：參數相同但生成權重可以不同，因forward函數不同。

**結果判讀與常見錯誤**：初始化尺度、LR與電路保持固定；若一起改，不能解釋是哪項改動的效果。

**理解問題**：為何不在這輪同時改電路？

<details><summary>提示與解答</summary>

先隔離mapping非線性這一個變因。

</details>


In [ ]:
torch.manual_seed(11)
affine=QuantumTrain(3,mapping_activation='affine')
torch.manual_seed(11)
tanh=QuantumTrain(3,mapping_activation='tanh')
for a,b in zip(affine.parameters(),tanh.parameters()):
    torch.testing.assert_close(a,b)
print(affine.mapping); print(tanh.mapping)
print('Same initial parameters; generated weights identical?',torch.allclose(affine.generated_weights(),tanh.generated_weights()))

## 3. 小資料過擬合與完整訓練

**本節目標**：先測能否記住每類2筆，共20筆。

**為什麼需要這一步**：連小資料都學不起來時，先檢查優化與表示能力，不急著擴大資料。

**輸入／輸出與 shape**：tiny固定20筆、full固定train split；只提供validation，不傳test。

**關鍵原理與程式定位**：run_mapping_study：smoke兩種資料規模各2epochs、單seed；full為tiny200epochs＋完整train20epochs，各3seeds。

**操作步驟**：執行凍結矩陣；正式可能需數分鐘。既有第一輪結果不改寫。

**預期觀察**：保存12個full runs或4個smoke runs、每epoch觀察與參數delta。

**結果判讀與常見錯誤**：tiny不能保證100%訓練accuracy；max_train_accuracy只供記憶能力診斷，正式checkpoint仍依validation loss。

**理解問題**：把tiny結果的高訓練accuracy當泛化能力可以嗎？

<details><summary>提示與解答</summary>

不行；它只檢查能否擬合固定小資料。

</details>


In [ ]:
study=run_mapping_study(OUTPUT/'mapping-studies',smoke=MODE=='smoke')
summary=pd.read_csv(study/'summary.csv')
display(summary)
print('Artifacts:',study)

## 4. 看梯度與activation隨時間變化

**本節目標**：分辨初始化檢查與整段訓練。

**為什麼需要這一步**：有些模型開始能更新，後來卻停滯。

**輸入／輸出與 shape**：history每epoch一列，gradient norm為batch平均，delta為epoch首尾差。

**關鍵原理與程式定位**：梯度、參數delta、ReLU零比例與logits跨圖片標準差搭配解讀。

**操作步驟**：畫完整train情境seed11的兩組曲線；可改成tiny觀察。

**預期觀察**：兩條loss和gradient曲線来自本輪run，不混入舊結果。

**結果判讀與常見錯誤**：epoch首尾delta可能因更新抵銷而小；不能直接等同每step沒有更新。

**理解問題**：量子梯度很小是否一定是barren plateau？

<details><summary>提示與解答</summary>

不是；先排除電路對称、資料、尺度及mapping。

</details>


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for activation in ('affine','tanh'):
    record=json.loads((study/f'{activation}-seed11-full'/'diagnostic.json').read_text())
    h=pd.DataFrame(record['history'])
    axes[0].plot(h.epoch,h.train_loss,label=activation)
    axes[1].plot(h.epoch,h.quantum_grad_norm,label=activation)
    axes[2].plot(h.epoch,[r['relu_zero_fraction'] for r in h.observation],label=activation)
for ax,label in zip(axes,['Train cross entropy','Quantum gradient norm (batch mean)','ReLU zero fraction on train probe']):
    ax.set(xlabel='Epoch',ylabel=label); ax.legend()
plt.tight_layout(); plt.show()
print('圖說：固定seed11，不是跨seed平均；train_loss在epoch結束後重新評估。')

## 5. 預測、權重與logits的變化

**本節目標**：檢查模型是否學到輸入差異。

**為什麼需要這一步**：單一accuracy可能掩蓋class collapse或不活躍hidden units。

**輸入／輸出與 shape**：initial/best/final observation → 權重統計、logits分布、prediction histogram。

**關鍵原理與程式定位**：best由validation loss決定；final是最後epoch，兩者不能混用。

**操作步驟**：比較best checkpoint的預測和初始預測，查看每個hidden的活躍率。

**預期觀察**：可能觀察到非線性改善，也可能仍學不動；依實際輸出說明。

**結果判讀與常見錯誤**：probe僅40筆train，分布不能直接代表完整validation或test。

**理解問題**：全部預測同一類就能證明程式有bug嗎？

<details><summary>提示與解答</summary>

不能，可能是表示或優化問題；搭配小資料實驗與梯度檢查定位。

</details>


In [ ]:
for activation in ('affine','tanh'):
    record=json.loads((study/f'{activation}-seed11-full'/'diagnostic.json').read_text())
    print(activation,'best epoch:',record['best_epoch'])
    display(pd.DataFrame({'initial':record['initial']['weights'],'best':record['best_observation']['weights']}))
    display(record['best_observation'])

## 6. 統計與結論的界線

**本節目標**：從驗證結果提出下一個可檢驗假說。

**為什麼需要這一步**：一次改動有收益，不代表已證明量子部分有幫助。

**輸入／輸出與 shape**：各情境×activation×seed → 均值與樣本SD；smoke單seed無SD。

**關鍵原理與程式定位**：下一步才是固定量子參數消融。本轮沒有使用test，不產生新的test主張。

**操作步驟**：彙整表格，寫出觀察、證據、限制各一句。

**預期觀察**：單seed std為NaN，表示N/A；full三seed只作描述統計。

**結果判讀與常見錯誤**：不能用本輪validation結果反覆調整後稱validation為獨立最終評估。

**理解問題**：Tanh較好是否代表量子電路提供優勢？

<details><summary>提示與解答</summary>

不能；需固定量子參數與純傳統生成器等消融比較。

</details>


In [ ]:
display(summary.groupby(['tiny','activation'])[['train_accuracy','val_accuracy','max_train_accuracy','train_seconds']].agg(['mean','std']))
print('Scope:', 'smoke pipeline only' if MODE=='smoke' else 'validation-only exploratory mapping comparison')
print('No test split was passed to diagnostic training.')